## Description
This script processes the compiled CAR data (from Krish) to create a clean, session-level dataset for Power BI analysis. It groups the data by "Contact Session ID" and aggregates key metrics such as call start time, starting hour, unique activity count, and total call duration in minutes. It also adds the call’s weekday and date for time-based analysis.The resulting CSV ("combined_calls_transformed_simple.csv") is optimized for Power BI visuals, enabling metrics like average call duration or activity patterns by hour and day.

In [1]:
import pandas as pd

# --- Step 1: Load the data ---
df = pd.read_csv("combined_calls.csv")

# --- Step 2: Ensure datetime format ---
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'], errors='coerce')

# --- Step 3: Group by Contact Session ID and aggregate ---
grouped = (
    df.groupby('Contact Session ID')
    .agg(
        Call_Start_Time=('Activity Start Timestamp', 'min'),   # earliest timestamp per call
        Starting_Hour=('hour', 'min'),                         # earliest hour per call
        Count=('Activity Start Timestamp', lambda x: len(set(x))),  # number of unique activity timestamps
        Call_Duration=('Activity Start Timestamp', 
                       lambda x: (max(x) - min(x)).total_seconds() / 60 if len(x) > 1 else 0)
    )
    .reset_index()
)

# --- Step 4: Add columns about day specifically  ---
grouped['DayOfWeekNum'] = grouped['Call_Start_Time'].dt.dayofweek + 1     # 1 = Monday, 7 = Sunday
grouped['Call_Start_Date'] = grouped['Call_Start_Time'].dt.date            # date only (no time)

# --- Step 5: Save for Power BI ---
grouped.to_csv("combined_calls_transformed_simple.csv", index=False)

display(grouped.head())


C:\Users\julia\AppData\Local\Temp\ipykernel_6276\660812848.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("combined_calls.csv")


,Contact Session ID,Call_Start_Time,Starting_Hour,Count,Call_Duration,DayOfWeekNum,Call_Start_Date
0,00002422-f51f-458b-82d6-cfa5a3f36fd9,2025-03-13 12:51:21,12,4,0.900000,4,2025-03-13
1,0000a8d5-cecb-46b1-82cf-b7ce07d85b24,2025-03-17 16:53:15,16,5,0.933333,1,2025-03-17
2,00011655-35de-476f-9a8c-dd48ed4d914a,2024-11-06 14:39:01,14,10,4.133333,3,2024-11-06
3,00014a58-a6ce-4cb2-a529-d55e2c9c304d,2025-02-28 08:33:40,8,8,2.283333,5,2025-02-28
4,00015327-f646-462f-a585-0552331eed4e,2025-06-03 07:43:56,7,3,0.200000,2,2025-06-03
